# Polarization

The polarization of the CMB is described by the Stokes parameters $Q$ and $U$. Unlike the
temperature $T$ (or intensity $I$), they are not scalars: they depend on the orientation of
the local reference frame, and the combinations $Q \pm iU$ transform as spin-2 fields on the
sphere. Their spin-weighted harmonic transform splits them into two scalar components,
the gradient-like **E modes** and the curl-like **B modes**:

$$
Q \pm iU = -\sum_{\ell m} \left(a^E_{\ell m} \pm i\, a^B_{\ell m}\right) {}_{\pm 2}Y_{\ell m}.
$$

jax-healpy follows the healpy conventions for these transforms. This page covers:

- simulating polarized maps from power spectra with `synfast`;
- the polarized transforms between $T, Q, U$ maps and $T, E, B$ alms;
- the spin transforms `map2alm_spin` and `alm2map_spin`;
- the six $TT, EE, BB, TE, EB, TB$ power spectra with `anafast`;
- the leakage of E modes into B modes on a masked sky;
- smoothing, the agreement with healpy, just-in-time compilation and differentiation.

!!! note
    The spherical harmonic transforms require the `recommended` extra (s2fft), see the
    [installation](installation.md) page.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from healpy import mollview
from matplotlib.colors import LinearSegmentedColormap

import jax_healpy as hp

# Enable 64-bit precision (recommended, jax-healpy warns otherwise)
jax.config.update('jax_enable_x64', True)

jax-healpy does not provide visualization functions: we display the maps with healpy's
`mollview`, in the Mollweide projection. A small helper draws several maps side by side,
with a color scale symmetric around zero.

In [ ]:
# Diverging colormap: blue (negative), gray (zero), red (positive).
CMAP = LinearSegmentedColormap.from_list('diverging', ['#104281', '#2a78d6', '#f0efec', '#e34948', '#9e1c1c'])


def mollview_grid(maps, titles, ncols=3):
    """Draw several HEALPix maps in µK side by side."""
    nrows = (len(maps) + ncols - 1) // ncols
    fig = plt.figure(figsize=(4.5 * ncols, 3 * nrows))
    for i, (m, title) in enumerate(zip(maps, titles)):
        vmax = float(jnp.abs(m).max())
        mollview(
            m,
            fig=fig.number,
            sub=(nrows, ncols, i + 1),
            title=title,
            cmap=CMAP,
            min=-vmax,
            max=vmax,
            unit='µK',
            format='%.3g',
        )
    plt.show()

## Simulating polarized maps

We use the lensed CMB power spectra of the Planck 2018 best-fit ΛCDM model, with a
tensor-to-scalar ratio $r = 0.01$. The file [`data/cmb_cls.txt`](data/cmb_cls.txt) gives
the $C_\ell$ of $TT, EE, BB, TE$ in µK², computed with CAMB by the script
`data/make_cls.py`. The $B$ modes come from the primordial gravitational waves at large
scales and from the gravitational lensing of the $E$ modes at small scales. The
polarization spectra vanish at $\ell < 2$, since a spin-2 field has no monopole or dipole.

With `lmax = 3 * nside - 1`, the default of the transforms, the spectra are arrays of
`lmax + 1` values.

In [ ]:
nside = 64
lmax = 3 * nside - 1

ell, cl_tt, cl_ee, cl_bb, cl_te = np.loadtxt('data/cmb_cls.txt')[: lmax + 1].T
# The spectra are usually plotted as D_ell = ell (ell + 1) C_ell / 2 pi
dl = ell * (ell + 1) / (2 * np.pi)

Given a list of four spectra, [`synfast`][jax_healpy.synfast] draws correlated $T, E, B$
alms and returns the $T, Q, U$ maps, stacked along the first axis. As in healpy,
`new=True` means that the spectra are ordered as $TT, EE, BB, TE$. Like the other random
functions of JAX, `synfast` takes a PRNG key as first argument.

In [ ]:
key = jax.random.key(0)
maps, alms = hp.synfast(key, [cl_tt, cl_ee, cl_bb, cl_te], nside, lmax=lmax, new=True, alm=True)

print(f'Maps: {maps.shape}')  # (T, Q, U) x npix
print(f'Alms: {alms.shape}')  # (T, E, B) x (lmax + 1) x (2 * lmax + 1)

In [ ]:
mollview_grid(maps, ['T', 'Q', 'U'])

The temperature fluctuates by about 100 µK, the polarization by about 1 µK only at this
resolution.

The alms use the s2fft layout by default: a two-dimensional array indexed by $\ell$ and by
$m + \ell_\mathrm{max}$, which also stores the negative $m$. Pass `healpy_ordering=True` to
the transforms to work with the one-dimensional healpy layout instead.

## From maps to alms

Given a stack of three maps, [`map2alm`][jax_healpy.map2alm] interprets them as $I, Q, U$
and returns the $T, E, B$ alms.

In [ ]:
alms_teb = hp.map2alm(maps, lmax=lmax)
print(f'Alms: {alms_teb.shape}')

As in healpy, this transform is only approximate: the HEALPix grid has no exact quadrature
rule, and the `iter` parameter (3 by default) sets the number of iterations that refine
the alms. Above $\ell \approx 2\,n_\mathrm{side}$, the recovered alms differ from the input
ones by a few percent.

Unlike healpy, jax-healpy also accepts a stack of two maps, interpreted as $Q, U$ only, and
then returns the $E, B$ alms. This avoids transforming a temperature map when only the
polarization is needed.

In [ ]:
alms_eb = hp.map2alm(maps[1:], lmax=lmax)
print(f'Alms: {alms_eb.shape}')
print(f'Same E, B alms as from T, Q, U maps: {jnp.allclose(alms_eb, alms_teb[1:])}')

## Spin transforms

[`map2alm_spin`][jax_healpy.map2alm_spin] and [`alm2map_spin`][jax_healpy.alm2map_spin]
are the spin-weighted transforms on which the polarized transforms are built. They take a
list of two maps or alms and the spin, 2 for polarization. With `spin=2`, `map2alm_spin`
gives the same $E, B$ alms as `map2alm`. Note that `map2alm_spin` does not iterate by
default (`iter=0`), as in healpy.

In [ ]:
alm_e, alm_b = hp.map2alm_spin([maps[1], maps[2]], spin=2, lmax=lmax)
alms_eb_no_iter = hp.map2alm(maps[1:], lmax=lmax, iter=0)
print(f'Same alms as map2alm(iter=0): {jnp.allclose(jnp.stack([alm_e, alm_b]), alms_eb_no_iter)}')

The spin transforms also handle other spins, for example spin 1 for the gradient of a
scalar field, or spin 0, in which case they act on a single map and are equivalent to
`map2alm` and `alm2map` for a temperature map.

## From alms to maps

[`alm2map`][jax_healpy.alm2map] is the inverse of `map2alm`: it turns a stack of $T, E, B$
alms into $T, Q, U$ maps, or a stack of $E, B$ alms into $Q, U$ maps.

To see what E and B modes look like, we keep only the E modes, then only the B modes, of
the simulated alms. A pure E-mode map has no B-mode power, and vice versa.

In [ ]:
alm_zero = jnp.zeros_like(alms[1])
q_e, u_e = hp.alm2map(jnp.stack([alms[1], alm_zero]), nside, lmax=lmax)
q_b, u_b = hp.alm2map_spin([alm_zero, alms[2]], nside, spin=2, lmax=lmax)

mollview_grid(
    [q_e, u_e, q_b, u_b], ['Q, E modes only', 'U, E modes only', 'Q, B modes only', 'U, B modes only'], ncols=2
)

cl_e_only = hp.anafast(jnp.stack([q_e, u_e]), lmax=lmax)
cl_b_only = hp.anafast(jnp.stack([q_b, u_b]), lmax=lmax)
print(f'E-mode maps: EE power = {cl_e_only[0].sum():.2e}, BB power = {cl_e_only[1].sum():.2e}')
print(f'B-mode maps: EE power = {cl_b_only[0].sum():.2e}, BB power = {cl_b_only[1].sum():.2e}')

The leakage between E and B is small but not zero, since `anafast` relies on the
approximate `map2alm` transform.

## Power spectra

For $I, Q, U$ maps, [`anafast`][jax_healpy.anafast] returns the six auto- and
cross-spectra, in the healpy order $TT, EE, BB, TE, EB, TB$. For $Q, U$ maps, it returns
the three spectra $EE, BB, EB$.

We compare them with the input spectra, drawn as dashed lines. The shaded bands show the
cosmic variance, the irreducible scatter of the spectra measured on a full sky, from the
$2\ell + 1$ modes available at each multipole: for an auto-spectrum,
$\sigma(C_\ell) = \sqrt{2 / (2\ell + 1)}\, C_\ell$. The auto-spectra span five orders of
magnitude and use a logarithmic scale. The $EB$ and $TB$ spectra vanish in this model: we
plot them in units of their standard deviation.

In [ ]:
cls = hp.anafast(maps, lmax=lmax)
print(f'Spectra: {cls.shape}')

COLORS = ['#2a78d6', '#eb6834', '#1baf7a']
nmodes = 2 * ell + 1
sigma_te = np.sqrt((cl_te**2 + cl_tt * cl_ee) / nmodes)
sigma_eb = np.sqrt(cl_ee * cl_bb / nmodes)
sigma_tb = np.sqrt(cl_tt * cl_bb / nmodes)

_, (ax_auto, ax_te, ax_null) = plt.subplots(1, 3, figsize=(15, 4.2))
s = slice(2, None)  # no polarization at ell < 2

for color, name, cl_out, cl_in in zip(COLORS, ['TT', 'EE', 'BB'], cls[:3], [cl_tt, cl_ee, cl_bb]):
    sigma = np.sqrt(2 / nmodes) * cl_in
    ax_auto.fill_between(ell[s], (dl * (cl_in - sigma))[s], (dl * (cl_in + sigma))[s], color=color, alpha=0.2, lw=0)
    ax_auto.plot(ell[s], (dl * cl_in)[s], color=color, lw=1, ls='--')
    ax_auto.plot(ell[s], (dl * cl_out)[s], color=color, lw=1.5, label=name)
ax_auto.set_yscale('log')
ax_auto.set_ylabel(r'$D_\ell$ [µK²]')
ax_auto.set_title('Auto-spectra')

ax_te.fill_between(ell[s], (dl * (cl_te - sigma_te))[s], (dl * (cl_te + sigma_te))[s], color=COLORS[0], alpha=0.2, lw=0)
ax_te.plot(ell[s], (dl * cl_te)[s], color=COLORS[0], lw=1, ls='--')
ax_te.plot(ell[s], (dl * cls[3])[s], color=COLORS[0], lw=1.5, label='TE')
ax_te.set_ylabel(r'$D_\ell$ [µK²]')
ax_te.set_title('TE cross-spectrum')

ax_null.axhspan(-1, 1, color='gray', alpha=0.2, lw=0)
ax_null.plot(ell[s], (cls[4] / sigma_eb)[s], color=COLORS[1], lw=1, label='EB')
ax_null.plot(ell[s], (cls[5] / sigma_tb)[s], color=COLORS[2], lw=1, label='TB')
ax_null.set_ylabel(r'$C_\ell / \sigma(C_\ell)$')
ax_null.set_title('EB, TB cross-spectra')

for ax in (ax_auto, ax_te, ax_null):
    ax.set_xlabel(r'$\ell$')
    ax.grid(alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False)
plt.show()

The measured spectra agree with the input ones within the cosmic variance, which is larger
at low $\ell$, where fewer modes are available. The $EB$ and $TB$ spectra are compatible
with zero.

## E-to-B leakage on a masked sky

On the full sky, the decomposition into E and B modes is unique. Real observations mask
part of the sky, such as the Galactic plane, and on a cut sky some modes are both E and B:
the much larger E modes then leak into the B modes. We mask the pixels within 20° of the
equator, as a stand-in for a Galactic mask, and measure the spectra of the E-mode maps
above. The pseudo-spectra of the masked maps are divided by the observed sky fraction
$f_\mathrm{sky}$, which corrects their amplitude to first order.

In [ ]:
theta, _ = hp.pix2ang(nside, jnp.arange(hp.nside2npix(nside)))
mask = jnp.abs(theta - jnp.pi / 2) > np.radians(20)
fsky = mask.mean()
print(f'Observed sky fraction: {fsky:.2f}')

qu_e = jnp.stack([q_e, u_e])
cl_full = hp.anafast(qu_e, lmax=lmax)  # EE, BB, EB
cl_masked = hp.anafast(qu_e * mask, lmax=lmax) / fsky

_, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(ell[s], (dl * cl_ee)[s], color='gray', lw=1, ls='--', label='EE input')
ax.plot(ell[s], (dl * cl_bb)[s], color=COLORS[2], lw=1, ls='--', label='BB input (lensing + r = 0.01)')
ax.plot(ell[s], (dl * cl_masked[1])[s], color=COLORS[1], lw=1.5, label='BB of the E modes, masked sky')
ax.plot(ell[s], (dl * cl_full[1])[s], color=COLORS[0], lw=1.5, label='BB of the E modes, full sky')
ax.set_yscale('log')
ax.set_xlabel(r'$\ell$')
ax.set_ylabel(r'$D_\ell$ [µK²]')
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.show()

On the full sky, the leakage only comes from the approximate `map2alm` and is negligible. On
the masked sky, the E modes leaking into B exceed the B-mode signal by more than an order of
magnitude at large scales, where the signal of the primordial gravitational waves is
sought. Apodizing the mask reduces this leakage, and "pure" pseudo-spectrum estimators
([Smith 2006](https://arxiv.org/abs/astro-ph/0511629)) remove it. jax-healpy does not
provide them, but its differentiable transforms are the building blocks of such estimators,
and of likelihoods that model the mask directly.
## Smoothing

`synfast`, `alm2map` and [`smoothing`][jax_healpy.smoothing] accept a Gaussian beam,
specified by its full width at half maximum `fwhm` in radians. For polarization, the beam
window of the E and B modes includes the spin-2 correction. Here, a 3° beam reduces the
amplitude of the $Q$ map:

In [ ]:
fwhm = float(np.radians(3))
maps_smoothed = hp.alm2map(alms, nside, lmax=lmax, fwhm=fwhm)

print(f'Standard deviation of Q: {maps[1].std():.2f} µK -> {maps_smoothed[1].std():.2f} µK after smoothing')
mollview_grid([maps[1], maps_smoothed[1]], ['Q', 'Q, smoothed by a 3° beam'], ncols=2)

## Agreement with healpy

jax-healpy follows the conventions of healpy and reproduces its results. We transform the
same alms, in the healpy layout, with both libraries:

In [ ]:
from healpy import alm2map as healpy_alm2map
from healpy import map2alm as healpy_map2alm

alms_hp = hp.synalm(key, [cl_tt, cl_ee, cl_bb, cl_te], lmax=lmax, new=True, healpy_ordering=True)
maps_jax = hp.alm2map(alms_hp, nside, lmax=lmax, healpy_ordering=True)
maps_healpy = healpy_alm2map(np.asarray(alms_hp), nside, lmax=lmax)
for name, m_jax, m_healpy in zip('TQU', maps_jax, maps_healpy):
    print(f'alm2map, {name}: relative RMS difference = {np.std(m_jax - m_healpy) / np.std(m_healpy):.1e}')

alms_jax = hp.map2alm(maps_healpy, lmax=lmax, healpy_ordering=True)
alms_healpy = healpy_map2alm(maps_healpy, lmax=lmax)
for name, a_jax, a_healpy in zip('TEB', alms_jax, alms_healpy):
    print(
        f'map2alm, {name}: relative RMS difference = {np.linalg.norm(a_jax - a_healpy) / np.linalg.norm(a_healpy):.1e}'
    )

In 64-bit precision, the two libraries agree to about $10^{-12}$, close to the accuracy of
the floating-point arithmetic. In 32-bit precision, the relative differences grow with the
resolution: about $10^{-4}$ at `nside = 64` and $10^{-3}$ at `nside = 256`, where they reach
10% for the B modes recovered by `map2alm`, much fainter than the E modes. Hence the
recommendation to enable 64-bit precision.

## JIT compilation and differentiation

The polarized transforms are JAX functions: they can be compiled with `jax.jit` and
differentiated with `jax.grad`. The parameters that set the shape of the arrays, such as
`nside`, `lmax` or the spin, must be static.

As an example, we compute the B-mode power of a pair of $Q, U$ maps and its gradient with
respect to the maps, which costs about as much as the function itself. Such gradients are
what gradient-based optimizers and samplers, such as Hamiltonian Monte Carlo, need to fit
models of the sky to maps, without approximating the transforms.

In [ ]:
@jax.jit
def b_mode_power(qu):
    _, alm_b = hp.map2alm_spin([qu[0], qu[1]], spin=2, lmax=lmax)
    return jnp.sum(jnp.abs(alm_b) ** 2)


qu = maps[1:]
power, gradient = jax.value_and_grad(b_mode_power)(qu)
print(f'B-mode power: {power:.3e}')
print(f'Gradient: {gradient.shape}')

# One step of gradient descent reduces the B-mode power.
step = 0.1 * power / jnp.sum(gradient**2)
print(f'B-mode power after one step: {b_mode_power(qu - step * gradient):.3e}')